# Fine-tune Laya on the synthetic Kannada set (#14)

Runs `training/finetune_laya.py`: Laya's multilingual checkpoint (at Laya's reviewed commit) is fine-tuned with Laya's own training loop on the 2,874 caller sentences in `Dhwanidesk/dhwani-desk-data`, asked the eight questions the backend asks Laya, worded exactly as it asks them. It is then exported to ONNX with the upstream exporter, checked against PyTorch, and uploaded to `Dhwanidesk/dhwani-desk-laya` with a tag the backend pins.

**Before running**
- Kaggle: Settings → Accelerator **GPU T4 x2** (one GPU is used) or P100, Internet **on**; Add-ons → Secrets → `HF_TOKEN` (a **write** token for the Dhwanidesk organization).
- Colab: Runtime → **T4 GPU**; Secrets (key icon) → `HF_TOKEN`, notebook access on.

Step 1 is a 10-minute smoke run through every step except the upload, so an export problem shows up before the long run. Step 2 is the real run, about 1–2 hours.

In [ ]:
BRANCH = 'main'  # or a feature branch, to run changes before they merge
!git clone --depth 1 -b {BRANCH} https://github.com/Pranav-0710/dhwani-desk.git
%cd dhwani-desk/training
# laya: the trainer; onnx, onnxscript, onnxruntime: the exporter and its check against PyTorch.
!pip install -q 'laya==0.4.6' onnx onnxscript onnxruntime huggingface_hub

In [ ]:
import os

import torch

try:
    from kaggle_secrets import UserSecretsClient

    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except ImportError:
    from google.colab import userdata

    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

## Step 1: smoke run, 64 rows, no upload

Trains one epoch on 64 rows, exports and verifies the ONNX graph. `--tag-dataset` tags the dataset repo's current head `v1.0` if that tag does not exist yet, so this run and the real one train on the same, pinned data (#12).

In [ ]:
!python finetune_laya.py --limit 64 --epochs 1 --tag-dataset --out out/smoke

## Step 2: full run and upload

Set `TAG` to a tag the model repo does not have yet. The script refuses to overwrite one. When it finishes, set `DECIDER=laya` and `LAYA_CHECKPOINT_REVISION` to the tag in the backend's `.env`.

In [ ]:
TAG = 'kn-ft-v1'
!python finetune_laya.py --out out/full --push-tag {TAG}

In [ ]:
import json

run = json.load(open('out/full/run.json', encoding='utf-8'))
print('rows', run['rows'], '| minutes', run['minutes'], '| loss per epoch', run['summary']['epoch_loss'])
for name, entry in run['summary']['calibration'].items():
    print(name, {k: v for k, v in entry.items() if k != 'issues'}, entry['issues'] or '')